# 🎯 Sign Language Recognition with ASL2000 Pretrained Weights

## 📋 Notebook Execution Order:

### **Phase 1: Setup & Data Preparation** (Cells 1-6)
1. **Cell 1**: Setup environment & directories
2. **Cell 2**: Download WLASL100 dataset from Kaggle
3. **Cell 3**: Load WLASL JSON metadata
4. **Cell 4**: Download videos from YouTube
5. **Cell 5**: Create train/val/test splits
6. **Cell 6**: Batch preprocess videos (~30 min)

### **Phase 2: Model & Training** (Cells 7-10)
7. **Cell 7a**: Load ASL2000 checkpoint (~5 sec)
8. **Cell 7b**: Create I3D model + load ASL2000 weights (~10 sec)
9. **Cell 8**: Create PyTorch Dataset & DataLoaders (~5 sec)
10. **Cell 10**: **Fine-tune on WLASL100** (~2 hours, target 70-75% acc)

### **Phase 3: ASL Citizen Cross-Dataset Training** (Cells 11a-11f)
11. **Cell 11a**: Load Citizen dataset
12. **Cell 11b**: Download Citizen videos
13. **Cell 11c**: Preprocess Citizen videos (~45 min)
14. **Cell 11d**: Create Citizen DataLoader
15. **Cell 11e**: Verify combined dataset
16. **Cell 11f**: **Fine-tune on Citizen** (~2 hours, target 75-80% acc)

---

## 🚀 Quick Start (Kaggle):

**Add these datasets as inputs:**
1. `dxiomt/wlasl` - WLASL100 videos
2. `wsasl2000-weights` - ASL2000 pretrained I3D model
3. `aslcitizen` - ASL Citizen dataset (optional, for Phase 3)

**Run cells in order:** 1 → 2 → 3 → 4 → 5 → 6 → 7a → 7b → 8 → 10

**Total time:** ~2.5 hours (preprocessing + training)

---

## 📊 Expected Results:

| Stage | Validation Accuracy | Notes |
|-------|-------------------|-------|
| Random init | ~1% | Baseline (100 classes) |
| ASL2000 weights | 40-50% | First epoch (pretrained features) |
| WLASL100 fine-tuning | **70-75%** | After Cell 10 |
| + Citizen fine-tuning | **75-80%** | After Cell 11f (cross-dataset) |

---

## ⚠️ Important Notes:

- **GPU Required**: T4 or better (16GB VRAM)
- **ASL2000 Architecture**: I3D (Inception3D), not Slow R50
- **Preprocessing**: Must run Cell 6 every time (data lost on session end)
- **Checkpoints**: Download best model after training!

---

In [1]:
# ---------- Cell 1: Setup Environment (Kaggle) ----------
import os
import sys

# Set base directory (Kaggle has /kaggle/working as workspace)
BASE_DIR = "/kaggle/working/WASL"
os.makedirs(BASE_DIR, exist_ok=True)

# Create subdirectories
os.makedirs(os.path.join(BASE_DIR, "videos"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "manifests"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "preprocessed"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "models"), exist_ok=True)

print("✅ Base directory:", BASE_DIR)
print("✅ Created subdirectories:")
print("   - videos/")
print("   - manifests/")
print("   - preprocessed/")
print("   - models/")

# Check GPU availability
import torch
if torch.cuda.is_available():
    print(f"\n✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️ No GPU detected - make sure GPU accelerator is enabled")
    print("   (Settings → Accelerator → GPU)")

print("\n✅ Ready to download WLASL dataset")

✅ Base directory: /kaggle/working/WASL
✅ Created subdirectories:
   - videos/
   - manifests/
   - preprocessed/
   - models/

✅ GPU available: Tesla T4
   GPU Memory: 15.83 GB

✅ Ready to download WLASL dataset


In [2]:
# ---------- Cell 2: Download WLASL Processed Dataset ----------
import zipfile
import shutil
from pathlib import Path

# Check if dataset is already added as input
KAGGLE_INPUT = "/kaggle/input/wlasl-processed"
if os.path.exists(KAGGLE_INPUT):
    print("✅ WLASL dataset found in Kaggle inputs!")
    print(f"   Path: {KAGGLE_INPUT}")
    
    # List available files
    print("\n📂 Available files:")
    for item in os.listdir(KAGGLE_INPUT):
        item_path = os.path.join(KAGGLE_INPUT, item)
        if os.path.isdir(item_path):
            print(f"   📁 {item}/")
        else:
            size_mb = os.path.getsize(item_path) / (1024 * 1024)
            print(f"   📄 {item} ({size_mb:.2f} MB)")
    
    # Check if videos folder exists
    videos_input = os.path.join(KAGGLE_INPUT, "videos")
    if os.path.exists(videos_input):
        video_count = len([f for f in os.listdir(videos_input) if f.endswith('.mp4')])
        print(f"\n✅ Found {video_count} videos in dataset")
    
    VIDEOS_SOURCE = videos_input
    
else:
    print("⚠️ Dataset not found in inputs!")
    print("\n📝 To add the dataset:")
    print("   1. Click 'Add Data' button (top right)")
    print("   2. Search for 'wlasl-processed'")
    print("   3. Add 'risangbaskoro/wlasl-processed' dataset")
    print("   4. Rerun this cell")
    sys.exit(1)

print("\n✅ Dataset ready - proceed to next cell")

✅ WLASL dataset found in Kaggle inputs!
   Path: /kaggle/input/wlasl-processed

📂 Available files:
   📄 nslt_2000.json (1.08 MB)
   📁 videos/
   📄 nslt_1000.json (0.67 MB)
   📄 WLASL_v0.3.json (11.38 MB)
   📄 wlasl_class_list.txt (0.02 MB)
   📄 nslt_300.json (0.26 MB)
   📄 missing.txt (0.05 MB)
   📄 nslt_100.json (0.10 MB)

✅ Found 11980 videos in dataset

✅ Dataset ready - proceed to next cell


In [3]:
# ---------- Cell 3: Load WLASL Manifest & Filter WLASL100 ----------
import json
import pandas as pd

# Load the main WLASL manifest
manifest_path = os.path.join(KAGGLE_INPUT, "WLASL_v0.3.json")
with open(manifest_path, "r") as f:
    wlasl_data = json.load(f)

print(f"✅ Loaded WLASL manifest: {len(wlasl_data)} total glosses")

# Filter for WLASL100 (first 100 glosses by frequency)
wlasl100_data = wlasl_data[:100]
print(f"✅ Filtered to WLASL100: {len(wlasl100_data)} glosses")

# Extract all video instances for WLASL100
video_records = []
for gloss_entry in wlasl100_data:
    gloss = gloss_entry['gloss']
    for instance in gloss_entry['instances']:
        video_id = instance['video_id']
        bbox = instance.get('bbox', None)
        fps = instance.get('fps', 25)
        frame_start = instance.get('frame_start', None)
        frame_end = instance.get('frame_end', None)
        split = instance.get('split', 'train')  # train/val/test
        
        video_records.append({
            'video_id': video_id,
            'gloss': gloss,
            'split': split,
            'bbox': bbox,
            'fps': fps,
            'frame_start': frame_start,
            'frame_end': frame_end
        })

# Create DataFrame
df = pd.DataFrame(video_records)
print(f"\n✅ Created dataset with {len(df)} video instances")

# Check split distribution
print("\n📊 Split distribution:")
print(df['split'].value_counts())

print("\n📊 Top 10 glosses:")
print(df['gloss'].value_counts().head(10))

# Save manifest to working directory
manifest_save_path = os.path.join(BASE_DIR, "manifests", "wlasl100_manifest.csv")
df.to_csv(manifest_save_path, index=False)
print(f"\n✅ Saved manifest to: {manifest_save_path}")

print("\n✅ Ready for next cell - video verification")

✅ Loaded WLASL manifest: 2000 total glosses
✅ Filtered to WLASL100: 100 glosses

✅ Created dataset with 2038 video instances

📊 Split distribution:
split
train    1442
val       338
test      258
Name: count, dtype: int64

📊 Top 10 glosses:
gloss
book        40
drink       35
computer    30
before      26
chair       26
go          26
clothes     25
who         25
candy       24
cousin      23
Name: count, dtype: int64

✅ Saved manifest to: /kaggle/working/WASL/manifests/wlasl100_manifest.csv

✅ Ready for next cell - video verification


In [4]:
# ---------- Cell 3b: Analyze Split Distribution Per Gloss ----------

# Check how one gloss (e.g., "book") is distributed
book_df = df[df['gloss'] == 'book']
print("📊 Example: 'book' gloss distribution:")
print(book_df['split'].value_counts())
print(f"   Total: {len(book_df)} videos\n")

# Check another example
drink_df = df[df['gloss'] == 'drink']
print("📊 Example: 'drink' gloss distribution:")
print(drink_df['split'].value_counts())
print(f"   Total: {len(drink_df)} videos\n")

# Summary statistics
print("📊 Overall statistics:")
print(f"   • Total glosses: {df['gloss'].nunique()}")
print(f"   • Total videos: {len(df)}")
print(f"   • Avg videos per gloss: {len(df) / df['gloss'].nunique():.1f}")
print(f"\n   • Train videos: {len(df[df['split']=='train'])} ({len(df[df['split']=='train'])/len(df)*100:.1f}%)")
print(f"   • Val videos: {len(df[df['split']=='val'])} ({len(df[df['split']=='val'])/len(df)*100:.1f}%)")
print(f"   • Test videos: {len(df[df['split']=='test'])} ({len(df[df['split']=='test'])/len(df)*100:.1f}%)")

print("\n✅ Each of the 100 glosses has its videos split across train/val/test")
print("✅ This ensures the model sees each sign during training and is tested on unseen examples")

📊 Example: 'book' gloss distribution:
split
train    30
val       6
test      4
Name: count, dtype: int64
   Total: 40 videos

📊 Example: 'drink' gloss distribution:
split
train    25
val       6
test      4
Name: count, dtype: int64
   Total: 35 videos

📊 Overall statistics:
   • Total glosses: 100
   • Total videos: 2038
   • Avg videos per gloss: 20.4

   • Train videos: 1442 (70.8%)
   • Val videos: 338 (16.6%)
   • Test videos: 258 (12.7%)

✅ Each of the 100 glosses has its videos split across train/val/test
✅ This ensures the model sees each sign during training and is tested on unseen examples


In [5]:
# ---------- Cell 4: Verify Available Videos & Match with Manifest ----------
import os
from pathlib import Path

# Get list of all available video files
available_videos = set()
for video_file in os.listdir(VIDEOS_SOURCE):
    if video_file.endswith('.mp4'):
        # Extract video_id (filename without extension)
        video_id = video_file.replace('.mp4', '')
        available_videos.add(video_id)

print(f"✅ Found {len(available_videos)} available videos in dataset")

# Check which videos from manifest are actually available
df['video_available'] = df['video_id'].isin(available_videos)
df['video_path'] = df['video_id'].apply(
    lambda vid: os.path.join(VIDEOS_SOURCE, f"{vid}.mp4") if vid in available_videos else None
)

# Statistics
total_required = len(df)
total_available = df['video_available'].sum()
missing_count = total_required - total_available

print(f"\n📊 Video Availability:")
print(f"   • Required by manifest: {total_required}")
print(f"   • Available: {total_available} ({total_available/total_required*100:.1f}%)")
print(f"   • Missing: {missing_count} ({missing_count/total_required*100:.1f}%)")

# Check availability by split
print(f"\n📊 Availability by split:")
for split_name in ['train', 'val', 'test']:
    split_df = df[df['split'] == split_name]
    available = split_df['video_available'].sum()
    total = len(split_df)
    print(f"   • {split_name}: {available}/{total} ({available/total*100:.1f}%)")

# Filter to only available videos
df_available = df[df['video_available'] == True].copy()
print(f"\n✅ Working dataset: {len(df_available)} videos across {df_available['gloss'].nunique()} glosses")

# Save filtered manifest
filtered_manifest_path = os.path.join(BASE_DIR, "manifests", "wlasl100_available.csv")
df_available.to_csv(filtered_manifest_path, index=False)
print(f"✅ Saved available videos manifest to: {filtered_manifest_path}")

# Check if any glosses lost all videos
glosses_with_videos = df_available['gloss'].value_counts()
original_glosses = df['gloss'].nunique()
remaining_glosses = len(glosses_with_videos)

if remaining_glosses < original_glosses:
    print(f"\n⚠️ Warning: {original_glosses - remaining_glosses} glosses have no available videos")
else:
    print(f"\n✅ All {remaining_glosses} glosses have at least one video")

print("\n✅ Ready for next cell - video preprocessing")

✅ Found 11980 available videos in dataset

📊 Video Availability:
   • Required by manifest: 2038
   • Available: 1013 (49.7%)
   • Missing: 1025 (50.3%)

📊 Availability by split:
   • train: 748/1442 (51.9%)
   • val: 165/338 (48.8%)
   • test: 100/258 (38.8%)

✅ Working dataset: 1013 videos across 100 glosses
✅ Saved available videos manifest to: /kaggle/working/WASL/manifests/wlasl100_available.csv

✅ All 100 glosses have at least one video

✅ Ready for next cell - video preprocessing


In [6]:
# ---------- Cell 5: Video Preprocessing - Setup & Utilities ----------
import cv2
import numpy as np
from tqdm import tqdm
import torch

print("✅ OpenCV version:", cv2.__version__)
print("✅ PyTorch version:", torch.__version__)
print("✅ CUDA available:", torch.cuda.is_available())

# Define preprocessing parameters
PREPROCESS_CONFIG = {
    'target_fps': 25,           # Resample all videos to 25 fps
    'target_frames': 32,        # Extract 32 frames per video (standard for I3D)
    'target_size': (224, 224),  # Resize frames to 224x224 (I3D input size)
    'normalize': True,          # Normalize pixel values to [0, 1]
}

print("\n📋 Preprocessing Configuration:")
for key, value in PREPROCESS_CONFIG.items():
    print(f"   • {key}: {value}")

# Video loading utility function
def load_video_frames(video_path, target_frames=32, target_size=(224, 224)):
    """
    Load video and extract uniformly sampled frames.
    
    Args:
        video_path: Path to video file
        target_frames: Number of frames to extract
        target_size: Target spatial size (H, W)
    
    Returns:
        frames: numpy array of shape (T, H, W, C) - T=target_frames, C=3 (RGB)
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")
    
    # Get video properties
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    # Calculate frame indices to sample uniformly
    if total_frames < target_frames:
        # If video has fewer frames, repeat last frame
        indices = np.linspace(0, total_frames - 1, target_frames, dtype=int)
    else:
        # Sample uniformly across video duration
        indices = np.linspace(0, total_frames - 1, target_frames, dtype=int)
    
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        
        if ret:
            # Convert BGR (OpenCV) to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # Resize to target size
            frame = cv2.resize(frame, target_size)
            frames.append(frame)
        else:
            # If frame read fails, repeat last valid frame
            if len(frames) > 0:
                frames.append(frames[-1])
            else:
                # Create blank frame if no valid frames yet
                frames.append(np.zeros((target_size[0], target_size[1], 3), dtype=np.uint8))
    
    cap.release()
    
    # Stack frames into single array: (T, H, W, C)
    frames = np.stack(frames, axis=0)
    
    return frames, total_frames, fps


# Test the function on one video
print("\n🧪 Testing video loader on sample video...")
sample_row = df_available.iloc[0]
sample_video_path = sample_row['video_path']
sample_gloss = sample_row['gloss']

try:
    frames, original_frames, original_fps = load_video_frames(
        sample_video_path, 
        target_frames=PREPROCESS_CONFIG['target_frames'],
        target_size=PREPROCESS_CONFIG['target_size']
    )
    
    print(f"✅ Successfully loaded video for gloss: '{sample_gloss}'")
    print(f"   • Original: {original_frames} frames @ {original_fps:.1f} fps")
    print(f"   • Processed: {frames.shape[0]} frames @ {PREPROCESS_CONFIG['target_fps']} fps")
    print(f"   • Frame shape: {frames.shape[1:]} (H, W, C)")
    print(f"   • Pixel value range: [{frames.min()}, {frames.max()}]")
    print(f"   • Memory size: {frames.nbytes / 1024 / 1024:.2f} MB")
    
except Exception as e:
    print(f"❌ Error loading video: {e}")

print("\n✅ Video preprocessing utilities ready")
print("✅ Ready for next cell - batch preprocessing")

✅ OpenCV version: 4.12.0
✅ PyTorch version: 2.6.0+cu124
✅ CUDA available: True

📋 Preprocessing Configuration:
   • target_fps: 25
   • target_frames: 32
   • target_size: (224, 224)
   • normalize: True

🧪 Testing video loader on sample video...
✅ Successfully loaded video for gloss: 'book'
   • Original: 75 frames @ 30.0 fps
   • Processed: 32 frames @ 25 fps
   • Frame shape: (224, 224, 3) (H, W, C)
   • Pixel value range: [0, 255]
   • Memory size: 4.59 MB

✅ Video preprocessing utilities ready
✅ Ready for next cell - batch preprocessing


In [7]:
# ---------- Cell 6: Batch Preprocess All Videos ----------
# SMART SKIP: This cell automatically skips already-processed videos!
# Only new/missing videos will be processed.

import pickle
from pathlib import Path

# Create preprocessed data directories
PREPROCESSED_DIR = os.path.join(BASE_DIR, "preprocessed")
os.makedirs(os.path.join(PREPROCESSED_DIR, "train"), exist_ok=True)
os.makedirs(os.path.join(PREPROCESSED_DIR, "val"), exist_ok=True)
os.makedirs(os.path.join(PREPROCESSED_DIR, "test"), exist_ok=True)

print("📁 Preprocessed data will be saved to:")
print(f"   {PREPROCESSED_DIR}")

# Check existing preprocessed videos
existing_counts = {}
for split_name in ['train', 'val', 'test']:
    split_dir = os.path.join(PREPROCESSED_DIR, split_name)
    existing_counts[split_name] = len(list(Path(split_dir).glob("*.npz")))

total_existing = sum(existing_counts.values())
print(f"\n📊 Found {total_existing} already preprocessed videos:")
for split_name, count in existing_counts.items():
    print(f"   • {split_name}: {count} videos")

if total_existing > 0:
    print(f"\n💡 These videos will be SKIPPED (fast!)")
    print(f"   Only new/missing videos will be processed.")

# Function to preprocess and save videos for one split
def preprocess_split(df_split, split_name):
    """
    Preprocess all videos in a split and save as compressed .npz files (uint8).
    
    STORAGE OPTIMIZATION:
    - Store as uint8 (0-255) instead of float32: 75% less space
    - Use .npz compression: additional 40% savings
    - Normalize on-the-fly during training (no speed loss)
    
    Expected storage: ~2.5 GB (vs 9.5 GB with old method)
    
    Args:
        df_split: DataFrame containing videos for this split
        split_name: 'train', 'val', or 'test'
    """
    print(f"\n{'='*60}")
    print(f"Processing {split_name.upper()} split: {len(df_split)} videos")
    print(f"{'='*60}")
    
    split_dir = os.path.join(PREPROCESSED_DIR, split_name)
    
    processed_records = []
    failed_videos = []
    skipped_videos = 0
    
    for idx, row in tqdm(df_split.iterrows(), total=len(df_split), desc=f"{split_name}"):
        video_id = row['video_id']
        video_path = row['video_path']
        gloss = row['gloss']
        
        # Check if video is already preprocessed
        save_path = os.path.join(split_dir, f"{video_id}.npz")
        
        if os.path.exists(save_path):
            # Skip processing - load metadata from existing file
            try:
                data = np.load(save_path)
                frames = data['frames']
                
                processed_records.append({
                    'video_id': video_id,
                    'gloss': gloss,
                    'split': split_name,
                    'save_path': save_path,
                    'original_frames': -1,  # Unknown (not saved in .npz)
                    'original_fps': -1,     # Unknown
                    'processed_frames': frames.shape[0],
                    'frame_shape': frames.shape[1:],
                })
                skipped_videos += 1
                continue  # Skip to next video
                
            except Exception as e:
                # If can't load existing file, reprocess it
                print(f"\n⚠️ Corrupted file {video_id}, reprocessing...")
        
        try:
            # Load and preprocess video
            frames, orig_frames, orig_fps = load_video_frames(
                video_path,
                target_frames=PREPROCESS_CONFIG['target_frames'],
                target_size=PREPROCESS_CONFIG['target_size']
            )
            
            # DO NOT normalize here - keep as uint8 (0-255) for storage efficiency
            # Normalization will happen on-the-fly during training
            # frames stays as uint8 dtype
            
            # Save as compressed .npz file (uint8 + gzip compression)
            np.savez_compressed(save_path, frames=frames)
            
            # Record metadata
            processed_records.append({
                'video_id': video_id,
                'gloss': gloss,
                'split': split_name,
                'save_path': save_path,
                'original_frames': orig_frames,
                'original_fps': orig_fps,
                'processed_frames': frames.shape[0],
                'frame_shape': frames.shape[1:],
            })
            
        except Exception as e:
            failed_videos.append({
                'video_id': video_id,
                'gloss': gloss,
                'error': str(e)
            })
            print(f"\n⚠️ Failed to process {video_id} ({gloss}): {e}")
    
    # Summary statistics
    print(f"\n✅ {split_name.upper()} split complete:")
    print(f"   • Total videos: {len(df_split)}")
    print(f"   • Skipped (already exists): {skipped_videos}")
    print(f"   • Newly processed: {len(processed_records) - skipped_videos}")
    print(f"   • Failed: {len(failed_videos)}")
    
    if len(failed_videos) > 0:
        print(f"\n⚠️ Failed videos saved to: {PREPROCESSED_DIR}/{split_name}_failed.txt")
        with open(os.path.join(PREPROCESSED_DIR, f"{split_name}_failed.txt"), 'w') as f:
            for fail in failed_videos:
                f.write(f"{fail['video_id']},{fail['gloss']},{fail['error']}\n")
    
    return processed_records, failed_videos


# Process each split
all_processed = {}
all_failed = {}

for split_name in ['train', 'val', 'test']:
    df_split = df_available[df_available['split'] == split_name].copy()
    processed, failed = preprocess_split(df_split, split_name)
    all_processed[split_name] = processed
    all_failed[split_name] = failed

# Create final preprocessed manifest
final_records = []
for split_name in ['train', 'val', 'test']:
    final_records.extend(all_processed[split_name])

df_preprocessed = pd.DataFrame(final_records)

# Save preprocessed manifest
preprocessed_manifest_path = os.path.join(BASE_DIR, "manifests", "wlasl100_preprocessed.csv")
df_preprocessed.to_csv(preprocessed_manifest_path, index=False)

print("\n" + "="*60)
print("PREPROCESSING COMPLETE")
print("="*60)
print(f"\n📊 Final Statistics:")
print(f"   • Total videos processed: {len(df_preprocessed)}")
print(f"   • Train: {len(all_processed['train'])}")
print(f"   • Val: {len(all_processed['val'])}")
print(f"   • Test: {len(all_processed['test'])}")
print(f"\n   • Total failed: {sum(len(all_failed[s]) for s in ['train', 'val', 'test'])}")

# Calculate disk space used
total_size = 0
for split_name in ['train', 'val', 'test']:
    split_dir = os.path.join(PREPROCESSED_DIR, split_name)
    for npy_file in Path(split_dir).glob("*.npy"):
        total_size += npy_file.stat().st_size

print(f"\n💾 Disk space used: {total_size / (1024**3):.2f} GB")
print(f"   • Average per video: {total_size / len(df_preprocessed) / (1024**2):.2f} MB")

print(f"\n✅ Preprocessed manifest saved to:")
print(f"   {preprocessed_manifest_path}")

print("\n✅ Ready for next cell - create PyTorch dataset")

📁 Preprocessed data will be saved to:
   /kaggle/working/WASL/preprocessed

📊 Found 1013 already preprocessed videos:
   • train: 748 videos
   • val: 165 videos
   • test: 100 videos

💡 These videos will be SKIPPED (fast!)
   Only new/missing videos will be processed.

Processing TRAIN split: 748 videos


train: 100%|██████████| 748/748 [00:19<00:00, 38.87it/s]



✅ TRAIN split complete:
   • Total videos: 748
   • Skipped (already exists): 748
   • Newly processed: 0
   • Failed: 0

Processing VAL split: 165 videos


val: 100%|██████████| 165/165 [00:04<00:00, 38.30it/s]



✅ VAL split complete:
   • Total videos: 165
   • Skipped (already exists): 165
   • Newly processed: 0
   • Failed: 0

Processing TEST split: 100 videos


test: 100%|██████████| 100/100 [00:02<00:00, 37.39it/s]


✅ TEST split complete:
   • Total videos: 100
   • Skipped (already exists): 100
   • Newly processed: 0
   • Failed: 0

PREPROCESSING COMPLETE

📊 Final Statistics:
   • Total videos processed: 1013
   • Train: 748
   • Val: 165
   • Test: 100

   • Total failed: 0

💾 Disk space used: 0.00 GB
   • Average per video: 0.00 MB

✅ Preprocessed manifest saved to:
   /kaggle/working/WASL/manifests/wlasl100_preprocessed.csv

✅ Ready for next cell - create PyTorch dataset


In [8]:
# ---------- Cell 7: Create I3D Model & Load ASL2000 Weights (FROM WLASL REPO) ----------
# 
# ⚠️  IMPORTANT: If you get "CUDA error: device-side assert triggered" when running this cell:
#     This means there's a cached CUDA error from a previous cell.
#     SOLUTION: Click "Session Options" (⚙️) → "Restart & Run All"

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# Clear CUDA cache and reset CUDA errors
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print("🔄 CUDA cache cleared")

print("="*60)
print("LOADING I3D MODEL (WLASL ARCHITECTURE)")
print("="*60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️  Using device: {device}")

# Download WLASL's I3D implementation directly
print("\n📦 Downloading I3D model code from WLASL repo...")
!wget -q https://raw.githubusercontent.com/dxli94/WLASL/master/code/I3D/pytorch_i3d.py -O /tmp/pytorch_i3d.py

import sys
sys.path.insert(0, '/tmp')

from pytorch_i3d import InceptionI3d

print("✅ I3D module loaded successfully (WLASL architecture)")

# Step 1: Create I3D model (RGB stream only)
print("\n🔄 Creating I3D model...")
model = InceptionI3d(
    num_classes=2000,  # Temporary, will change to 100
    spatial_squeeze=True,
    final_endpoint='Logits',
    in_channels=3,     # RGB
    dropout_keep_prob=0.5
)

print("✅ I3D model architecture created (matches ASL2000)")

# Step 2: Load ASL2000 weights
print(f"\n🔄 Loading ASL2000 weights...")

WEIGHTS_BASE = "/kaggle/input/wsasl2000-weights/asl2000"
pt_files = [f for f in os.listdir(WEIGHTS_BASE) if f.endswith('.pt')]
checkpoint_path = os.path.join(WEIGHTS_BASE, pt_files[0])

checkpoint = torch.load(checkpoint_path, map_location='cpu')

# Load state dict
missing, unexpected = model.load_state_dict(checkpoint, strict=False)
print(f"✅ Weights loaded!")
print(f"   • Missing keys: {len(missing)}")
print(f"   • Unexpected keys: {len(unexpected)}")

if len(missing) == 0:
    print(f"   ✅✅ PERFECT MATCH! All layers loaded!")
elif len(missing) < 10:
    print(f"   ✅ Good match! Only classifier layers missing (expected)")
else:
    print(f"   ⚠️  Warning: {len(missing)} layers missing")

# Verify weights are pretrained
sample_weights = list(checkpoint.values())[10]
weight_std = sample_weights.std().item()
print(f"   • Weight Std Dev: {weight_std:.6f}")

if weight_std > 0.05:
    print(f"   ✅ Weights are PRETRAINED (ASL2000)!")
else:
    print(f"   ⚠️  Weights look random (std too low)")

# Step 3: Replace final layer for 100 classes
print(f"\n🔄 Adapting model to 100 classes...")

# Use the model's built-in method to replace logits
model.replace_logits(num_classes=100)

print(f"✅ Adapted to 100 classes (WLASL100)")

# Step 4: Move to GPU
print(f"\n🔄 Moving model to GPU...")
try:
    model = model.to(device)
    print(f"✅ Model successfully moved to {device}")
except RuntimeError as e:
    if "CUDA error" in str(e):
        print(f"\n❌ CUDA Error detected!")
        print(f"   This is likely a cached error from previous cell execution.")
        print(f"\n🔧 SOLUTION:")
        print(f"   1. Click 'Session Options' (⚙️ top right)")
        print(f"   2. Click 'Restart & Run All'")
        print(f"   3. OR: Run this cell again after restart")
        raise
    else:
        raise

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Statistics:")
print(f"   • Total parameters: {total_params:,}")
print(f"   • Trainable parameters: {trainable_params:,}")
print(f"   • Model size: ~{total_params * 4 / 1e6:.1f} MB")

# Test forward pass to verify output shape
print(f"\n🧪 Testing forward pass...")
model.eval()
dummy_input = torch.randn(2, 3, 32, 224, 224).to(device)  # (B, C, T, H, W)
with torch.no_grad():
    dummy_output = model(dummy_input)
print(f"   • Input shape: {dummy_input.shape}")
print(f"   • Output shape (raw): {dummy_output.shape}")

# I3D with spatial_squeeze returns [batch, classes, time]
# We need to average over time dimension (dim=2)
if dummy_output.dim() == 3:
    dummy_output_avg = dummy_output.mean(dim=2)
    print(f"   • Output shape (after mean): {dummy_output_avg.shape}")
    print(f"   • Expected final: torch.Size([2, 100])")
    
    if dummy_output_avg.shape == torch.Size([2, 100]):
        print(f"   ✅ Output shape correct after temporal averaging!")
    else:
        print(f"   ⚠️  WARNING: Output shape mismatch!")
        print(f"   • Got: {dummy_output_avg.shape}")
        print(f"   • Expected: [2, 100]")
else:
    print(f"   • Expected: torch.Size([2, 100])")
    if dummy_output.shape == torch.Size([2, 100]):
        print(f"   ✅ Output shape correct!")
    else:
        print(f"   ⚠️  WARNING: Unexpected output dimensions!")

model.train()  # Back to training mode

print("\n" + "="*60)
print("✅ MODEL READY FOR FINE-TUNING")
print("="*60)

print(f"\n🎯 Training progression:")
print(f"   1. ✅ Kinetics-400 → I3D pretrained")
print(f"   2. ✅ WLASL2000 → ASL2000 weights (32.48% on 2000 classes)")
print(f"   3. 🔄 WLASL100 → Fine-tune (target: 70-75%)")

🔄 CUDA cache cleared
LOADING I3D MODEL (WLASL ARCHITECTURE)

🖥️  Using device: cuda

📦 Downloading I3D model code from WLASL repo...
✅ I3D module loaded successfully (WLASL architecture)

🔄 Creating I3D model...
✅ I3D model architecture created (matches ASL2000)

🔄 Loading ASL2000 weights...
✅ Weights loaded!
   • Missing keys: 0
   • Unexpected keys: 0
   ✅✅ PERFECT MATCH! All layers loaded!
   • Weight Std Dev: 0.442026
   ✅ Weights are PRETRAINED (ASL2000)!

🔄 Adapting model to 100 classes...
✅ Adapted to 100 classes (WLASL100)

🔄 Moving model to GPU...
✅ Model successfully moved to cuda

📊 Model Statistics:
   • Total parameters: 12,389,764
   • Trainable parameters: 12,389,764
   • Model size: ~49.6 MB

🧪 Testing forward pass...
   • Input shape: torch.Size([2, 3, 32, 224, 224])
   • Output shape (raw): torch.Size([2, 100, 3])
   • Output shape (after mean): torch.Size([2, 100])
   • Expected final: torch.Size([2, 100])
   ✅ Output shape correct after temporal averaging!

✅ MODEL 

In [9]:
# ---------- Cell 8: PyTorch Dataset & DataLoader ----------
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

print("✅ PyTorch version:", torch.__version__)
print("✅ CUDA available:", torch.cuda.is_available())

# Load preprocessed manifest
manifest_path = os.path.join(BASE_DIR, "manifests", "wlasl100_preprocessed.csv")
df_preprocessed = pd.read_csv(manifest_path)

print(f"\n✅ Loaded manifest: {len(df_preprocessed)} videos")
print(f"   • Train: {len(df_preprocessed[df_preprocessed['split']=='train'])}")
print(f"   • Val: {len(df_preprocessed[df_preprocessed['split']=='val'])}")
print(f"   • Test: {len(df_preprocessed[df_preprocessed['split']=='test'])}")

# Create label mapping (gloss name → integer label)
unique_glosses = sorted(df_preprocessed['gloss'].unique())
gloss_to_label = {gloss: idx for idx, gloss in enumerate(unique_glosses)}
label_to_gloss = {idx: gloss for gloss, idx in gloss_to_label.items()}

print(f"\n✅ Created label mapping for {len(unique_glosses)} classes")
print(f"\n📋 Sample labels:")
for i, (gloss, label) in enumerate(list(gloss_to_label.items())[:10]):
    print(f"   {label:2d} → {gloss}")
print("   ...")

# Add numeric labels to dataframe
df_preprocessed['label'] = df_preprocessed['gloss'].map(gloss_to_label)

# Save label mapping
label_map_path = os.path.join(BASE_DIR, "manifests", "label_mapping.json")
import json
with open(label_map_path, 'w') as f:
    json.dump({
        'gloss_to_label': gloss_to_label,
        'label_to_gloss': label_to_gloss,
        'num_classes': len(unique_glosses)
    }, f, indent=2)
print(f"\n✅ Saved label mapping to: {label_map_path}")


# PyTorch Dataset class
class WLASLDataset(Dataset):
    """
    PyTorch Dataset for WLASL preprocessed videos.
    
    Args:
        dataframe: DataFrame with columns ['save_path', 'label']
        transform: Optional transform to apply to video frames
        augment: Whether to apply data augmentation (for training)
    """
    
    def __init__(self, dataframe, transform=None, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.augment = augment
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Load preprocessed video frames (compressed .npz format)
        video_path = self.df.loc[idx, 'save_path']
        data = np.load(video_path)
        frames = data['frames']  # uint8, Shape: (T, H, W, C) = (32, 224, 224, 3)
        
        # Apply augmentation if training (on uint8 - faster)
        if self.augment:
            frames = self._augment_video(frames)
        
        # Normalize to [0, 1] NOW (on-the-fly during loading)
        frames = frames.astype(np.float32) / 255.0
        
        # Convert to PyTorch tensor: (C, T, H, W) - I3D expects this format
        frames = torch.from_numpy(frames).float()  # (T, H, W, C)
        frames = frames.permute(3, 0, 1, 2)  # (C, T, H, W)
        
        # Apply transform if provided
        if self.transform:
            frames = self.transform(frames)
        
        # Get label
        label = self.df.loc[idx, 'label']
        
        # Get metadata (for debugging/analysis)
        video_id = self.df.loc[idx, 'video_id']
        gloss = self.df.loc[idx, 'gloss']
        
        return {
            'frames': frames,      # (C, T, H, W) = (3, 32, 224, 224)
            'label': label,        # int (0-99)
            'video_id': video_id,  # string
            'gloss': gloss         # string
        }
    
    def _augment_video(self, frames):
        """
        Apply data augmentation to video frames (uint8 format).
        
        Args:
            frames: (T, H, W, C) numpy array, uint8, range [0, 255]
        
        Returns:
            Augmented frames (T, H, W, C), uint8, range [0, 255]
        """
        # Random horizontal flip (50% chance)
        # Note: Be careful with sign language - some signs change meaning when flipped!
        # For WLASL100, we'll use conservative augmentations
        if np.random.rand() > 0.5:
            frames = np.flip(frames, axis=2).copy()  # Flip width dimension
        
        # Random temporal cropping (simulate speed variation)
        # Keep 80-100% of frames, resample to 32
        if np.random.rand() > 0.3:
            T = frames.shape[0]
            start_ratio = np.random.uniform(0, 0.2)
            end_ratio = np.random.uniform(0.8, 1.0)
            start_idx = int(T * start_ratio)
            end_idx = int(T * end_ratio)
            
            # Extract subset and resample to 32 frames
            subset = frames[start_idx:end_idx]
            indices = np.linspace(0, len(subset)-1, 32, dtype=int)
            frames = subset[indices]
        
        # Random brightness/contrast (subtle) - FIXED: work on uint8 range
        if np.random.rand() > 0.5:
            brightness_factor = np.random.uniform(0.9, 1.1)
            frames = frames.astype(np.float32) * brightness_factor  # Convert to float
            frames = np.clip(frames, 0, 255)  # Clip to valid uint8 range [0, 255]
            frames = frames.astype(np.uint8)  # Convert back to uint8
        
        return frames


# Create datasets for each split
train_df = df_preprocessed[df_preprocessed['split'] == 'train'].copy()
val_df = df_preprocessed[df_preprocessed['split'] == 'val'].copy()
test_df = df_preprocessed[df_preprocessed['split'] == 'test'].copy()

train_dataset = WLASLDataset(train_df, augment=True)   # Augmentation ON for training
val_dataset = WLASLDataset(val_df, augment=False)       # Augmentation OFF for validation
test_dataset = WLASLDataset(test_df, augment=False)     # Augmentation OFF for testing

print("\n" + "="*60)
print("PYTORCH DATASETS CREATED")
print("="*60)
print(f"\n📊 Dataset sizes:")
print(f"   • Train: {len(train_dataset)} videos (with augmentation)")
print(f"   • Val:   {len(val_dataset)} videos (no augmentation)")
print(f"   • Test:  {len(test_dataset)} videos (no augmentation)")

# Create DataLoaders
BATCH_SIZE = 8  # Adjust based on GPU memory (T4 can handle 8-16)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,      # Shuffle for training
    num_workers=2,     # Parallel data loading (adjust based on CPU cores)
    pin_memory=True    # Faster GPU transfer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,     # No shuffle for validation
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,     # No shuffle for testing
    num_workers=2,
    pin_memory=True
)

print(f"\n📦 DataLoaders created:")
print(f"   • Batch size: {BATCH_SIZE}")
print(f"   • Train batches: {len(train_loader)} ({len(train_loader)*BATCH_SIZE} samples)")
print(f"   • Val batches:   {len(val_loader)} ({len(val_loader)*BATCH_SIZE} samples)")
print(f"   • Test batches:  {len(test_loader)} ({len(test_loader)*BATCH_SIZE} samples)")

# Test loading one batch
print("\n🧪 Testing data loading...")
sample_batch = next(iter(train_loader))

print(f"✅ Successfully loaded batch:")
print(f"   • Frames shape: {sample_batch['frames'].shape}")  # (B, C, T, H, W)
print(f"   • Labels shape: {sample_batch['label'].shape}")   # (B,)
print(f"   • Frames dtype: {sample_batch['frames'].dtype}")
print(f"   • Frames range: [{sample_batch['frames'].min():.3f}, {sample_batch['frames'].max():.3f}]")
print(f"\n   Sample batch glosses:")
for i in range(min(4, BATCH_SIZE)):
    print(f"      {i+1}. {sample_batch['gloss'][i]} (label={sample_batch['label'][i].item()})")

print("\n✅ Dataset and DataLoader ready for training!")
print("✅ Ready for next cell - load pretrained I3D model")

✅ PyTorch version: 2.6.0+cu124
✅ CUDA available: True

✅ Loaded manifest: 1013 videos
   • Train: 748
   • Val: 165
   • Test: 100

✅ Created label mapping for 100 classes

📋 Sample labels:
    0 → accident
    1 → africa
    2 → all
    3 → apple
    4 → basketball
    5 → bed
    6 → before
    7 → bird
    8 → birthday
    9 → black
   ...

✅ Saved label mapping to: /kaggle/working/WASL/manifests/label_mapping.json

PYTORCH DATASETS CREATED

📊 Dataset sizes:
   • Train: 748 videos (with augmentation)
   • Val:   165 videos (no augmentation)
   • Test:  100 videos (no augmentation)

📦 DataLoaders created:
   • Batch size: 8
   • Train batches: 94 (752 samples)
   • Val batches:   21 (168 samples)
   • Test batches:  13 (104 samples)

🧪 Testing data loading...
✅ Successfully loaded batch:
   • Frames shape: torch.Size([8, 3, 32, 224, 224])
   • Labels shape: torch.Size([8])
   • Frames dtype: torch.float32
   • Frames range: [0.000, 1.000]

   Sample batch glosses:
      1. woman (lab

In [10]:
# ---------- Cell 10: Training Loop (Official WLASL Config) ----------

import time
from datetime import datetime
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm

# Check if model exists
if 'model' not in dir():
    raise RuntimeError("❌ ERROR: Model not found! Run Cell 8 (Load I3D Model) first!")

# Check if dataloaders exist
if 'train_loader' not in dir() or 'val_loader' not in dir():
    raise RuntimeError("❌ ERROR: DataLoaders not found! Run Cell 9 (Create DataLoaders) first!")

print("✅ Model and DataLoaders found")

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("🔄 CUDA cache cleared")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Training config - OFFICIAL WLASL SETTINGS
config = {
    'num_epochs': 100,
    'learning_rate': 1e-4,  # Official WLASL
    'weight_decay': 1e-8,   # Official WLASL (NOT 1e-4!)
    'adam_eps': 1e-3,       # Official WLASL
    'patience': 10,
    'grad_clip': 1.0,
    'use_amp': False,       # WLASL doesn't use AMP
}

print("="*60)
print("TRAINING SETUP (OFFICIAL WLASL CONFIG)")
print("="*60)
print(f"Device: {device}")
print(f"Epochs: {config['num_epochs']}")
print(f"LR: {config['learning_rate']} | Weight Decay: {config['weight_decay']}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# Optimizer - OFFICIAL WLASL USES ADAM (NOT AdamW) with SAME LR for all layers
optimizer = optim.Adam(
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay'],
    eps=config['adam_eps']
)

# Learning rate scheduler - OFFICIAL WLASL USES ReduceLROnPlateau
from torch.optim.lr_scheduler import ReduceLROnPlateau

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=5,
    factor=0.3,
    verbose=True
)

criterion = nn.CrossEntropyLoss()

# Training tracking
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'learning_rates': []}
best_val_loss = float('inf')
best_val_acc = 0.0
patience_counter = 0
best_epoch = 0
checkpoint_dir = os.path.join(BASE_DIR, "models", "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)


def train_epoch(model, dataloader, criterion, optimizer, device, epoch):
    """Train for one epoch - Official WLASL style (no FP16, standard PyTorch)."""
    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Train]")
    
    for batch_idx, batch in enumerate(pbar):
        frames = batch['frames'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(frames)
        
        # I3D outputs: [batch, classes, time] - average over time
        if outputs.dim() == 3:
            outputs = outputs.mean(dim=2)
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        optimizer.step()
        
        # Calculate accuracy (move to CPU for computation)
        with torch.no_grad():
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            running_loss += loss.item()
        
        # Update progress bar
        avg_loss = running_loss / (batch_idx + 1)
        avg_acc = 100 * correct / total
        pbar.set_postfix({
            'loss': f'{avg_loss:.4f}',
            'acc': f'{avg_acc:.2f}%'
        })
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc


def validate(model, dataloader, criterion, device, epoch):
    """Validate on validation set - Official WLASL style."""
    model.eval()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Val]  ")
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(pbar):
            frames = batch['frames'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(frames)
            
            # I3D outputs: [batch, classes, time] - average over time
            if outputs.dim() == 3:
                outputs = outputs.mean(dim=2)
            
            loss = criterion(outputs, labels)
            
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            running_loss += loss.item()
            
            avg_loss = running_loss / (batch_idx + 1)
            avg_acc = 100 * correct / total
            pbar.set_postfix({
                'loss': f'{avg_loss:.4f}',
                'acc': f'{avg_acc:.2f}%'
            })
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc

# Main training loop
# Main training loop
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60 + "\n")

start_time = time.time()

for epoch in range(config['num_epochs']):
    epoch_start = time.time()
    
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device, epoch
    )
    
    # Validate
    val_loss, val_acc = validate(
        model, val_loader, criterion, device, epoch
    )
    
    # Update scheduler (ReduceLROnPlateau - needs val_loss)
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['learning_rates'].append(current_lr)
    
    # Calculate times
    epoch_time = time.time() - epoch_start
    elapsed_time = time.time() - start_time
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{config['num_epochs']} Summary:")
    print(f"{'='*60}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"Learning Rate: {current_lr:.6f}")
    print(f"Epoch Time: {epoch_time:.1f}s | Total Time: {elapsed_time/60:.1f}min")
    
    # GPU memory stats
    if epoch == 0:
        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        print(f"GPU Memory: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")
    
    # Check if best model
    is_best = val_acc > best_val_acc
    
    if is_best:
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_epoch = epoch + 1
        patience_counter = 0
        
        # Save best model
        best_model_path = os.path.join(checkpoint_dir, "best_model.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'history': history
        }, best_model_path)
        
        print(f"✅ New best model saved! (Val Acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"⏳ No improvement. Patience: {patience_counter}/{config['patience']}")
    
    print(f"🏆 Best Val Acc so far: {best_val_acc:.2f}% (Epoch {best_epoch})")
    print("="*60 + "\n")
    
    # Early stopping
    if patience_counter >= config['patience']:
        print(f"\n🛑 Early stopping triggered after {epoch+1} epochs")
        print(f"   No improvement for {config['patience']} epochs")
        print(f"   Best model from epoch {best_epoch} will be used")
        break
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history
        }, checkpoint_path)
        print(f"💾 Checkpoint saved: epoch_{epoch+1}.pth\n")
    
    # Clear cache every 3 epochs (helps with memory fragmentation)
    if (epoch + 1) % 3 == 0:
        torch.cuda.empty_cache()

# Training complete
total_time = time.time() - start_time

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\n📊 Final Results:")
print(f"   • Total epochs: {len(history['train_loss'])}")
print(f"   • Total time: {total_time/60:.1f} minutes")
print(f"   • Best validation accuracy: {best_val_acc:.2f}% (Epoch {best_epoch})")
print(f"   • Best validation loss: {best_val_loss:.4f}")
print(f"\n   • Final train accuracy: {history['train_acc'][-1]:.2f}%")
print(f"   • Final val accuracy: {history['val_acc'][-1]:.2f}%")

# Load best model
print(f"\n🔄 Loading best model (epoch {best_epoch})...")
best_checkpoint = torch.load(os.path.join(checkpoint_dir, "best_model.pth"))
model.load_state_dict(best_checkpoint['model_state_dict'])
print("✅ Best model loaded")

# Save training history
history_path = os.path.join(BASE_DIR, "models", "training_history.json")
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f"\n💾 Training history saved to: {history_path}")
print("\n✅ Ready for next cell - evaluation on test set")

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


✅ Model and DataLoaders found
🔄 CUDA cache cleared
TRAINING SETUP (OFFICIAL WLASL CONFIG)
Device: cuda
Epochs: 100
LR: 0.0001 | Weight Decay: 1e-08
Train batches: 94 | Val batches: 21

STARTING TRAINING



Epoch 1 [Val]  : 100%|██████████| 21/21 [00:10<00:00,  2.02it/s, loss=4.5591, acc=2.42%]



Epoch 1/100 Summary:
Train Loss: 4.6840 | Train Acc: 1.34%
Val Loss:   4.5591 | Val Acc:   2.42%
Learning Rate: 0.000100
Epoch Time: 133.7s | Total Time: 2.2min
GPU Memory: 0.24 GB allocated, 8.67 GB reserved
✅ New best model saved! (Val Acc: 2.42%)
🏆 Best Val Acc so far: 2.42% (Epoch 1)



Epoch 2 [Val]  : 100%|██████████| 21/21 [00:10<00:00,  2.02it/s, loss=4.4303, acc=4.85%]


...


Epoch 23/100 Summary:
Train Loss: 0.6026 | Train Acc: 94.79%
Val Loss:   1.0200 | Val Acc:   75.15%
Learning Rate: 0.000100
Epoch Time: 134.1s | Total Time: 51.6min
✅ New best model saved! (Val Acc: 75.15%)
🏆 Best Val Acc so far: 75.15% (Epoch 23)



Epoch 24 [Val]  : 100%|██████████| 21/21 [00:10<00:00,  2.01it/s, loss=0.9854, acc=72.12%]



Epoch 24/100 Summary:
Train Loss: 0.5116 | Train Acc: 95.05%
Val Loss:   0.9854 | Val Acc:   72.12%
Learning Rate: 0.000100
Epoch Time: 133.9s | Total Time: 53.8min
⏳ No improvement. Patience: 1/10
🏆 Best Val Acc so far: 75.15% (Epoch 23)



Epoch 25 [Train]:  31%|███       | 29/94 [00:40<01:30,  1.39s/it, loss=0.4120, acc=96.55%]


KeyboardInterrupt: 